In [1]:
from mpi4py import MPI
from dolfinx import mesh
#print(dolfinx.__version__)
import dolfinx

In [2]:
print(dolfinx.__file__)

/Users/yanjun/Documents/apps/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/dolfinx/__init__.py


In [3]:
domain = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8, mesh.CellType.quadrilateral)

In [4]:
from dolfinx.fem import functionspace
V = functionspace(domain, ('Lagrange',1))

In [5]:
from dolfinx import fem
uD = fem.Function(V)
uD.interpolate(lambda x: 1+x[0]**2 + 2* x[1]**2)

In [14]:
print(uD.x.array)

[1.       1.03125  1.015625 1.046875 1.125    1.140625 1.0625   1.09375
 1.28125  1.296875 1.1875   1.140625 1.171875 1.5      1.515625 1.34375
 1.265625 1.25     1.28125  1.78125  1.796875 1.5625   1.421875 1.375
 1.390625 1.421875 2.125    2.140625 1.84375  1.640625 1.53125  1.515625
 1.5625   1.59375  2.53125  2.546875 2.1875   1.921875 1.75     1.671875
 1.6875   1.765625 1.796875 3.       3.015625 2.59375  2.265625 2.03125
 1.890625 1.84375  1.890625 2.       2.03125  3.0625   2.671875 2.375
 2.171875 2.0625   2.046875 2.125    3.140625 2.78125  2.515625 2.34375
 2.265625 2.28125  3.25     2.921875 2.6875   2.546875 2.5      3.390625
 3.09375  2.890625 2.78125  3.5625   3.296875 3.125    3.765625 3.53125
 4.      ]


In [15]:
type(uD.x.array)

numpy.ndarray

In [16]:
import numpy
tdim = domain.topology.dim
fdim = tdim - 1
domain.topology.create_connectivity(fdim, tdim)


In [19]:
print(tdim)
print(fdim)

2
1


In [20]:
type(domain)

dolfinx.mesh.Mesh

In [32]:
print(domain.topology.create_connectivity)

<bound method Topology.create_connectivity of <dolfinx.mesh.Topology object at 0x118061010>>


In [33]:
boundary_facets = mesh.exterior_facet_indices(domain.topology)

In [34]:
type(boundary_facets)

numpy.ndarray

In [35]:
print(boundary_facets)

[  0   1   3   5   9  13  17  23  27  35  39  49  53  65  69  83  86  87
 100 101 102 113 114 123 124 131 132 137 138 141 142 143]


In [37]:
boundary_dofs = fem.locate_dofs_topological(V, fdim, boundary_facets)
type(boundary_dofs)

numpy.ndarray

In [38]:
print(boundary_dofs)

[ 0  1  2  4  6  8 11 13 17 19 24 26 32 34 41 43 44 51 52 53 59 60 65 66
 70 71 74 75 77 78 79 80]


In [40]:
bc = fem.dirichletbc(uD, boundary_dofs)
type(bc)

dolfinx.fem.bcs.DirichletBC

In [45]:
print(bc.dof_indices())

(array([ 0,  1,  2,  4,  6,  8, 11, 13, 17, 19, 24, 26, 32, 34, 41, 43, 44,
       51, 52, 53, 59, 60, 65, 66, 70, 71, 74, 75, 77, 78, 79, 80],
      dtype=int32), 32)


In [54]:
print(bc.g.x.array)

[1.       1.03125  1.015625 1.046875 1.125    1.140625 1.0625   1.09375
 1.28125  1.296875 1.1875   1.140625 1.171875 1.5      1.515625 1.34375
 1.265625 1.25     1.28125  1.78125  1.796875 1.5625   1.421875 1.375
 1.390625 1.421875 2.125    2.140625 1.84375  1.640625 1.53125  1.515625
 1.5625   1.59375  2.53125  2.546875 2.1875   1.921875 1.75     1.671875
 1.6875   1.765625 1.796875 3.       3.015625 2.59375  2.265625 2.03125
 1.890625 1.84375  1.890625 2.       2.03125  3.0625   2.671875 2.375
 2.171875 2.0625   2.046875 2.125    3.140625 2.78125  2.515625 2.34375
 2.265625 2.28125  3.25     2.921875 2.6875   2.546875 2.5      3.390625
 3.09375  2.890625 2.78125  3.5625   3.296875 3.125    3.765625 3.53125
 4.      ]


In [57]:
import ufl
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
type(u)

ufl.argument.Argument

In [63]:
print(u.number())

1


In [73]:
print(u.evaluate)

<bound method Terminal.evaluate of Argument(FunctionSpace(Mesh(blocked element (Basix element (P, quadrilateral, 1, gll_warped, unset, False, float64, []), (2,)), 0), Basix element (P, quadrilateral, 1, gll_warped, unset, False, float64, [])), 1, None)>


In [75]:
from dolfinx import default_scalar_type
f = fem.Constant(domain, default_scalar_type(-6))
type(f)

dolfinx.fem.function.Constant

In [84]:
print(f.value)

-6.0


In [85]:
a = ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
type(a)

ufl.form.Form

In [92]:
L = f * v * ufl.dx

In [103]:
from dolfinx.fem.petsc import LinearProblem
problem = LinearProblem(a, L, bcs=[bc], petsc_options={'ksp_type':'preonly','pc_type':'lu'})

In [104]:
type(problem)

dolfinx.fem.petsc.LinearProblem

In [115]:
print(problem.b.array)
print(problem.u.x.array)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [119]:
uh = problem.solve()

In [121]:
print(uh.x.array)

[1.       1.03125  1.015625 1.046875 1.125    1.140625 1.0625   1.09375
 1.28125  1.296875 1.1875   1.140625 1.171875 1.5      1.515625 1.34375
 1.265625 1.25     1.28125  1.78125  1.796875 1.5625   1.421875 1.375
 1.390625 1.421875 2.125    2.140625 1.84375  1.640625 1.53125  1.515625
 1.5625   1.59375  2.53125  2.546875 2.1875   1.921875 1.75     1.671875
 1.6875   1.765625 1.796875 3.       3.015625 2.59375  2.265625 2.03125
 1.890625 1.84375  1.890625 2.       2.03125  3.0625   2.671875 2.375
 2.171875 2.0625   2.046875 2.125    3.140625 2.78125  2.515625 2.34375
 2.265625 2.28125  3.25     2.921875 2.6875   2.546875 2.5      3.390625
 3.09375  2.890625 2.78125  3.5625   3.296875 3.125    3.765625 3.53125
 4.      ]
